In [ ]:
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
q("""
select table_name, column_name, data_type
from information_schema.columns
where table_schema = 'prime'
  and table_name in ('subscriptions', 'subscription_members', 'tariffs')
order by table_name, ordinal_position
""")

In [ ]:
print(q("""
select count(*) as строк,
       count(left_at) as вышли,
       count(*) - count(left_at) as остались
from prime.subscription_members
""").to_string(index=False))

print(q("""
select is_trial, count(*) as подписок,
       count(ended_at) as отменено
from prime.subscriptions
group by 1 order by 1
""").to_string(index=False))

print(q("""
select cancel_reason, count(*) as подписок
from prime.subscriptions
where ended_at is not null
group by 1 order by 2 desc
""").to_string(index=False))

In [ ]:
print(q("""
select count(*) filter (where m.joined_at < s.started_at)                as вступили_до_старта,
       count(*) filter (where m.left_at < m.joined_at)                   as вышли_до_вступления,
       count(*) filter (where s.ended_at is not null and m.left_at is null) as в_отменённой_без_выхода,
       count(*) filter (where m.left_at > s.ended_at)                    as вышли_после_отмены
from prime.subscription_members m
join prime.subscriptions s on s.subscription_id = m.subscription_id
""").T)

print(q("""
with m as (
    select subscription_id,
           count(*) as n_ever,
           count(*) filter (where left_at is null) as n_now
    from prime.subscription_members
    group by 1)
select n_ever, n_now, count(*) as подписок
from m group by 1, 2 order by 1, 2
""").to_string(index=False))

In [ ]:
# 1. Сколько участников когда-либо бывает по тарифам
print(q("""
select s.tariff_id, t.share_slots, m.n_ever, count(*) as подписок
from (select subscription_id, count(*) as n_ever
      from prime.subscription_members group by 1) m
join prime.subscriptions s using (subscription_id)
join (select tariff_id, max(share_slots) as share_slots
      from prime.tariffs group by 1) t using (tariff_id)
group by 1, 2, 3 order by 1, 3
""").to_string(index=False))

# 2. Выходы позже отмены: сколько и насколько позже
print(q("""
select count(*) as строк,
       count(distinct m.subscription_id) as подписок,
       round(avg(extract(epoch from (m.left_at - s.ended_at)) / 86400)::numeric, 1) as дней_в_среднем,
       max(extract(epoch from (m.left_at - s.ended_at)) / 86400)::int as дней_максимум
from prime.subscription_members m
join prime.subscriptions s using (subscription_id)
where m.left_at > s.ended_at
""").to_string(index=False))

In [ ]:
q("""
select column_name, data_type
from information_schema.columns
where table_schema = 'prime' and table_name = 'service_usage'
order by ordinal_position
""")

In [ ]:
cohort = q("""
with sl as (
    select tariff_id, max(share_slots) as share_slots
    from prime.tariffs group by 1),
m as (
    select m.subscription_id,
           count(*) filter (where m.joined_at < s.started_at + interval '90 days') as n90
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    group by 1)
select s.subscription_id, s.tariff_id, sl.share_slots, s.is_trial,
       s.cancel_reason, s.started_at, s.ended_at,
       coalesce(m.n90, 0) as n90,
       (s.ended_at is null or s.ended_at >= s.started_at + interval '90 days') as alive90,
       (s.ended_at is not null and s.ended_at < s.started_at + interval '180 days') as ended180
from prime.subscriptions s
join sl on sl.tariff_id = s.tariff_id
left join m on m.subscription_id = s.subscription_id
where s.started_at < '2026-02-20'
""")

print("строк:", len(cohort), "| ключ уникален:", cohort["subscription_id"].is_unique)
print("дожили до 90 дней:", int(cohort["alive90"].sum()))
print("участников у подписок больше лимита (должно быть 0):",
      int((cohort["n90"] > cohort["share_slots"]).sum()))

In [ ]:
print(q("""
select count(*) as участий,
       round(percentile_cont(0.5)  within group (order by extract(epoch from (m.joined_at - s.started_at))/86400)::numeric, 1) as p50_дней,
       round(percentile_cont(0.9)  within group (order by extract(epoch from (m.joined_at - s.started_at))/86400)::numeric, 1) as p90_дней,
       round(percentile_cont(0.99) within group (order by extract(epoch from (m.joined_at - s.started_at))/86400)::numeric, 1) as p99_дней,
       count(*) filter (where m.joined_at > s.ended_at) as после_отмены
from prime.subscription_members m
join prime.subscriptions s on s.subscription_id = m.subscription_id
""").to_string(index=False))

In [ ]:
base = cohort[cohort["alive90"]].copy()
base["full"] = base["n90"] == base["share_slots"]

print(base.groupby(["tariff_id", "full"]).agg(
    подписок=("subscription_id", "size"),
    доля_отмен_90_180=("ended180", "mean")).round(4).to_string())

print()
print(base.groupby(["tariff_id", "n90"]).agg(
    подписок=("subscription_id", "size"),
    доля_отмен_90_180=("ended180", "mean")).round(4).to_string())

In [ ]:
md = q("""
select m.subscription_id,
       extract(epoch from (m.joined_at - s.started_at))/86400 as day
from prime.subscription_members m
join prime.subscriptions s on s.subscription_id = m.subscription_id
where s.started_at < '2026-02-20'
""")
cohort["end_day"] = (cohort["ended_at"] - cohort["started_at"]).dt.total_seconds() / 86400

rows = []
for N in (30, 45, 60, 90):
    cnt = md[md["day"] < N].groupby("subscription_id").size().rename("n")
    d = cohort.join(cnt, on="subscription_id")
    d["n"] = d["n"].fillna(0)
    d = d[d["end_day"].isna() | (d["end_day"] >= N)]          # только дожившие до N
    d["full"] = d["n"] == d["share_slots"]
    d["ended"] = (d["end_day"].notna() & (d["end_day"] < N + 90)).astype(float)
    for t, g in d.groupby("tariff_id"):
        a, b = g.loc[g["full"], "ended"], g.loc[~g["full"], "ended"]
        se = (a.var() / len(a) + b.var() / len(b)) ** 0.5
        rows.append({"N": N, "tariff": t, "full_n": len(a), "rest_n": len(b),
                     "full": round(a.mean(), 4), "rest": round(b.mean(), 4),
                     "diff_pp": round((a.mean() - b.mean()) * 100, 2),
                     "ci95_pp": f"±{1.96 * se * 100:.2f}"})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
ended = base[base["ended180"]]
print(pd.crosstab([ended["tariff_id"], ended["full"]],
                  ended["cancel_reason"], normalize="index").round(3).to_string())